# Data Cleaning - Airline Passenger Satisfaction

Uses `src/preprocessing.clean_data` with logging. Raw CSVs untouched. Children (Age<10, 1894 rows 1.8%) retained. Outliers logged not removed. Arrival Delay median from train reused for test (no leakage).

In [1]:
import sys
from pathlib import Path
# allow import from project root when notebook is in notebooks/
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np, logging, glob
import src.logger
from src.preprocessing import clean_data, preprocess_data, load_data
log=logging.getLogger(__name__)
df_train_raw, df_test_raw = load_data()
print('raw train',df_train_raw.shape,'test',df_test_raw.shape)
print('Age<10 in raw train', (df_train_raw['Age']<10).sum())

C:\Users\DIXIT\Desktop\airline-passenger-satisfaction\src\train.py:29: UserWarning: XGBoost not installed — skipping XGBClassifier.
  warnings.warn("XGBoost not installed — skipping XGBClassifier.")


raw train (103904, 25) test (25976, 25)
Age<10 in raw train 1894


In [2]:
print('Missing before train')
print(df_train_raw.isnull().sum().to_string())
print('dups train', df_train_raw.duplicated().sum(), 'test', df_test_raw.duplicated().sum())
display(df_train_raw.describe().T)

Missing before train
Unnamed: 0                             0
id                                     0
Gender                                 0
Customer Type                          0
Age                                    0
Type of Travel                         0
Class                                  0
Flight Distance                        0
Inflight wifi service                  0
Departure/Arrival time convenient      0
Ease of Online booking                 0
Gate location                          0
Food and drink                         0
Online boarding                        0
Seat comfort                           0
Inflight entertainment                 0
On-board service                       0
Leg room service                       0
Baggage handling                       0
Checkin service                        0
Inflight service                       0
Cleanliness                            0
Departure Delay in Minutes             0
Arrival Delay in Minutes            

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,103904.0,51951.500000,29994.645522,0.0,25975.75,51951.5,77927.25,103903.0
id,103904.0,64924.210502,37463.812252,1.0,32533.75,64856.5,97368.25,129880.0
Age,103904.0,39.379706,15.114964,7.0,27.00,40.0,51.00,85.0
Flight Distance,103904.0,1189.448375,997.147281,31.0,414.00,843.0,1743.00,4983.0
Inflight wifi service,103904.0,2.729683,1.327829,0.0,2.00,3.0,4.00,5.0
Departure/Arrival time convenient,103904.0,3.060296,1.525075,0.0,2.00,3.0,4.00,5.0
Ease of Online booking,103904.0,2.756901,1.398929,0.0,2.00,3.0,4.00,5.0
Gate location,103904.0,2.976883,1.277621,0.0,2.00,3.0,4.00,5.0
Food and drink,103904.0,3.202129,1.329533,0.0,2.00,3.0,4.00,5.0
Online boarding,103904.0,3.250375,1.349509,0.0,2.00,3.0,4.00,5.0


In [3]:
df_train_clean = clean_data(df_train_raw)
# reuse train median for test to avoid leakage
df_test_clean = clean_data(df_test_raw, arrival_median=df_train_raw['Arrival Delay in Minutes'].median())
print('clean train',df_train_clean.shape)
print('clean test',df_test_clean.shape)
print('missing after train', int(df_train_clean.isnull().sum().sum()), 'test', int(df_test_clean.isnull().sum().sum()))
print('children retained train', int((df_train_clean['Age']<10).sum()))
display(df_train_clean.head())

clean train (103904, 23)
clean test (25976, 23)
missing after train 0 test 0
children retained train 1894


,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,3,1,...,5,4,3,4,4,5,5,25,18,neutral or dissatisfied
1,Male,disloyal Customer,25,Business travel,Business,235,3,2,3,3,...,1,1,5,3,1,4,1,1,6,neutral or dissatisfied
2,Female,Loyal Customer,26,Business travel,Business,1142,2,2,2,2,...,5,4,3,4,4,4,5,0,0,satisfied
3,Female,Loyal Customer,25,Business travel,Business,562,2,5,5,5,...,2,2,5,3,1,4,2,11,9,neutral or dissatisfied
4,Male,Loyal Customer,61,Business travel,Business,214,3,3,3,3,...,3,3,4,4,3,3,3,0,0,satisfied


In [4]:
for c in ['Flight Distance','Departure Delay in Minutes','Arrival Delay in Minutes']:
    q1,q3=df_train_clean[c].quantile([0.25,0.75]); iqr=q3-q1
    out=int(((df_train_clean[c]<q1-1.5*iqr)|(df_train_clean[c]>q3+1.5*iqr)).sum())
    print(c, 'outliers', out, f'({out/len(df_train_clean)*100:.2f}%) - retained')
X_train,y_train,X_test,y_test = preprocess_data(df_train_raw, df_test_raw)
print('X_train',X_train.shape,'y',y_train.value_counts().to_dict())
print('X_test',X_test.shape)
print('cols dropped: Unnamed: 0, id - target encoded 1/0 - raw CSVs still 103904x25')

Flight Distance outliers 2291 (2.20%) - retained
Departure Delay in Minutes outliers 14529 (13.98%) - retained
Arrival Delay in Minutes outliers 13954 (13.43%) - retained


X_train (103904, 22) y {0: 58879, 1: 45025}
X_test (25976, 22)
cols dropped: Unnamed: 0, id - target encoded 1/0 - raw CSVs still 103904x25


In [5]:
import pathlib
lf=sorted(glob.glob(str(ROOT/'logs/*.log')))[-1]
print(lf)
print(open(lf,encoding='utf-8').read()[-2500:])

C:\Users\DIXIT\Desktop\airline-passenger-satisfaction\logs\08_28_2026_11_14_06.log
[2026-08-28 11:14:07,116] line 1 __main__ - INFO - test

